In [ ]:
import httpx, pandas as pd, numpy as np, os
from dotenv import load_dotenv

load_dotenv("../.env")

CACHE = "history.parquet"
PAGE_SIZE = 1000
DEFCON_FIRST_SEASON = "2025-26" #defensive_contribution didn't exist before this, older 0s mean missing not zero

def fetch_all_gameweeks(c):
    #short page signals the end, same as app/player_client.py
    rows, offset = [], 0
    while True:
        r = c.get("/player-gameweeks", params={"limit": PAGE_SIZE, "offset": offset})
        r.raise_for_status()
        page = r.json()
        rows.extend(page)
        if len(page) < PAGE_SIZE:
            return rows
        offset += PAGE_SIZE

def build_history():
    headers = {"X-API-Key": os.environ["PLAYER_SERVICE_API_KEY"]}
    with httpx.Client(base_url="http://localhost:8000", headers=headers, timeout=30) as c:
        df = pd.DataFrame(fetch_all_gameweeks(c))

    df.loc[df["season"] < DEFCON_FIRST_SEASON, "defensive_contribution"] = np.nan
    return df

if os.path.exists(CACHE):
    df = pd.read_parquet(CACHE)
else:
    df = build_history()
    df.to_parquet(CACHE)

In [ ]:
POSITIONS = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
played = df[df["minutes"] > 0]

print(f"rows: {len(df):,}  players: {df['player_code'].nunique():,}")
print(f"share of 0-minute rows: {(df['minutes'] == 0).mean():.1%}")

#points by position - all rows vs only rows where they played, since 0-minute rows drag the mean down
by_pos = pd.DataFrame({
    "rows": df.groupby("element_type").size(),
    "mean_pts": df.groupby("element_type")["total_points"].mean(),
    "mean_pts_played": played.groupby("element_type")["total_points"].mean(),
    "std_pts_played": played.groupby("element_type")["total_points"].std(),
    "zero_min_share": df.groupby("element_type")["minutes"].apply(lambda m: (m == 0).mean()),
}).rename(index=POSITIONS)
display(by_pos.round(2))

display(played[["total_points", "minutes", "expected_goals", "expected_assists", "bps", "price", "fixture_difficulty"]]
        .describe().T.round(2))

In [ ]:
#season drift - scoring rules change (defcon points arrived in 2025-26), so check before pooling seasons
season_pos = played.pivot_table(index="season", columns="element_type", values="total_points", aggfunc="mean").rename(columns=POSITIONS)
season_pos["played_rows"] = played.groupby("season").size()
display(season_pos.round(2))

#points distribution for rows where they played - how rare are hauls
bins = [-10, 0, 1, 2, 3, 5, 8, 12, 100]
labels = ["<=0", "1", "2", "3", "4-5", "6-8", "9-12", "13+"]
dist = pd.crosstab(pd.cut(played["total_points"], bins=bins, labels=labels), played["element_type"].map(POSITIONS), normalize="columns")
display((dist * 100).round(1).rename_axis(index="points", columns="% of played rows"))

In [ ]:
#lagged features per player within a season - shift(1) first so a row's features never include its own gameweek
gw = df.sort_values(["player_code", "season", "round"]).copy()
g = gw.groupby(["player_code", "season"])
gw["xgi"] = gw["expected_goals"] + gw["expected_assists"]
gw["played"] = (gw["minutes"] > 0).astype(int)
gw["started"] = (gw["starts"] > 0).astype(int)
gw["gw_index"] = g.cumcount() #how many earlier gameweeks this player has in the season

for col in ["total_points", "minutes", "xgi", "bps", "played", "started"]:
    prev = g[col].shift(1)
    gw[f"{col}_last1"] = prev
    for n in (3, 5):
        gw[f"{col}_last{n}"] = prev.groupby([gw["player_code"], gw["season"]]).transform(lambda s: s.rolling(n, min_periods=1).mean())

#minutes persistence - the first stage of a two-stage model lives or dies on this
persist = gw.dropna(subset=["played_last1"]).groupby("played_last1")["played"].mean()
print(f"P(plays | didn't play last GW) = {persist[0]:.1%}")
print(f"P(plays | played last GW)      = {persist[1]:.1%}")

full3 = gw[gw["gw_index"] >= 3] #only rows with a full 3-GW window
starts_in_last3 = (full3["started_last3"] * 3).round().astype(int)
display(full3.groupby(starts_in_last3)[["played", "started"]].mean()
        .assign(rows=full3.groupby(starts_in_last3).size())
        .rename(columns={"played": "P(plays)", "started": "P(starts)"})
        .rename_axis("starts in last 3").round(3))

In [ ]:
#which recent-form signals actually predict this GW's points, among players who played (spearman - points are skewed)
lag_cols = [c for c in gw.columns if c.endswith(("_last1", "_last3", "_last5"))]
feature_cols = lag_cols + ["price", "fixture_difficulty"]
played_gw = gw[(gw["played"] == 1) & (gw["gw_index"] >= 5)]
corr = pd.DataFrame({
    pos: played_gw[played_gw["element_type"] == code][feature_cols]
         .corrwith(played_gw[played_gw["element_type"] == code]["total_points"], method="spearman")
    for code, pos in POSITIONS.items()
})
corr["all"] = played_gw[feature_cols].corrwith(played_gw["total_points"], method="spearman")
display(corr.sort_values("all", ascending=False).round(3))

In [ ]:
#fixture context among single-fixture GWs where they played - difficulty and home/away by position
single = played[played["fixture_count"] == 1]
display(single.pivot_table(index="fixture_difficulty", columns="element_type", values="total_points", aggfunc="mean")
        .rename(columns=POSITIONS).assign(rows=single.groupby("fixture_difficulty").size()).round(2))
display(single.pivot_table(index="was_home", columns="element_type", values="total_points", aggfunc="mean")
        .rename(columns=POSITIONS).round(2))

#double gameweeks - roughly double the points?
display(played.groupby("fixture_count")["total_points"].agg(["mean", "size"]).round(2))

In [ ]:
#window length sweep - windows run across seasons (player_code is stable) so long windows aren't cut off every august
WINDOWS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30, 38]
HALFLIVES = [2, 4, 8, 16] #exponentially weighted alternative - recent games count more instead of a hard cutoff
MAX_WINDOW = max(WINDOWS)

def rolling_means(frame, col):
    #shift(1) so a row's window only ever covers earlier gameweeks
    by_player = frame.groupby("player_code")[col].shift(1).groupby(frame["player_code"])
    out = {n: by_player.rolling(n, min_periods=n).mean().reset_index(level=0, drop=True) for n in WINDOWS}
    out.update({f"ewm hl={h}": by_player.ewm(halflife=h).mean().reset_index(level=0, drop=True) for h in HALFLIVES})
    return pd.DataFrame(out, index=frame.index)

hist = df.sort_values(["player_code", "season", "round"]).reset_index(drop=True)
hist["prior_gws"] = hist.groupby("player_code").cumcount()
apps = hist[hist["minutes"] > 0].copy()
apps["prior_apps"] = apps.groupby("player_code").cumcount()

pts_per_gw = rolling_means(hist, "total_points")   #0-minute weeks count as 0 - mixes in "does he play"
mins_per_gw = rolling_means(hist, "minutes")
pts_per_app = rolling_means(apps, "total_points")  #only games they actually played - pure form

#same rows for every window: played, with a full 38-GW and 38-appearance history
eval_idx = apps.index[(apps["prior_apps"] >= MAX_WINDOW) & (hist.loc[apps.index, "prior_gws"] >= MAX_WINDOW)]
target = hist.loc[eval_idx, "total_points"]
print(f"eval rows: {len(eval_idx):,}  players: {hist.loc[eval_idx, 'player_code'].nunique():,}")

sweep = pd.DataFrame({
    "pts / gameweek": pts_per_gw.loc[eval_idx].corrwith(target, method="spearman"),
    "pts / appearance": pts_per_app.loc[eval_idx].corrwith(target, method="spearman"),
    "minutes / gameweek": mins_per_gw.loc[eval_idx].corrwith(target, method="spearman"),
}).rename_axis("window")
display(sweep.round(3))

#same sweep for pts / appearance split by position
pos_of = hist.loc[eval_idx, "element_type"].map(POSITIONS)
display(pd.DataFrame({
    pos: pts_per_app.loc[eval_idx[pos_of == pos]].corrwith(target[pos_of == pos], method="spearman")
    for pos in POSITIONS.values()
}).rename_axis("window").round(3))
print(f"rows per position: {pos_of.value_counts().to_dict()}")